In [ ]:
# Cell 1: Imports & Config
import os, sys, json, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import OrderedDict

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    f1_score, precision_score, recall_score, matthews_corrcoef,
    average_precision_score, confusion_matrix
)
from lightgbm import LGBMClassifier
from imblearn.ensemble import BalancedBaggingClassifier
try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False

PROJECT_ROOT = os.path.dirname(os.path.abspath("__file__"))
if not os.path.exists(os.path.join(PROJECT_ROOT, "config.py")):
    PROJECT_ROOT = os.path.dirname(os.getcwd())
if not os.path.exists(os.path.join(PROJECT_ROOT, "config.py")):
    PROJECT_ROOT = os.getcwd()
    while PROJECT_ROOT != "/" and not os.path.exists(os.path.join(PROJECT_ROOT, "config.py")):
        PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

from config import SEED
import src.columns_real as CR

PI_TEST = 0.20
FINAL_BENIGN_FRAC = 0.80
N_BOOT = 50
BOOT_SEED = 123
PANEL_SPLIT_FRAC = 0.50
HIGH_MISS_THR = 0.50

RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results', 'v30_missing_handling_rejudge')
os.makedirs(RESULTS_DIR, exist_ok=True)
REPORTS_DIR = os.path.join(PROJECT_ROOT, 'reports')
os.makedirs(REPORTS_DIR, exist_ok=True)

ID_COL = CR.ID_COL
TARGET = CR.TARGET_COL

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"SEED={SEED}, PI_TEST={PI_TEST}, N_BOOT={N_BOOT}")
print(f"Results -> {RESULTS_DIR}")

# ============================================================================

In [ ]:
# Cell 2: Veri Yukleme + Ortak Sutun Temizligi (4 panel, NB39/NB46 ile ayni)
# ============================================================================
DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'real_data')

def load_panel(name):
    return pd.read_csv(os.path.join(DATA_DIR, CR.PANEL_INFO[name]["file"]))

df_master = load_panel("MASTER")
df_kanser = load_panel("KANSER")
df_cftr = load_panel("CFTR")
df_pah = load_panel("PAH")

print(f"MASTER: {df_master.shape} (pos={df_master[TARGET].sum()}, neg={(df_master[TARGET]==0).sum()})")
print(f"KANSER: {df_kanser.shape} (pos={df_kanser[TARGET].sum()}, neg={(df_kanser[TARGET]==0).sum()})")
print(f"CFTR:   {df_cftr.shape}   (pos={df_cftr[TARGET].sum()}, neg={(df_cftr[TARGET]==0).sum()})")
print(f"PAH:    {df_pah.shape}  (pos={df_pah[TARGET].sum()}, neg={(df_pah[TARGET]==0).sum()})")

feat_cols_raw = [c for c in df_master.columns if c not in [ID_COL, TARGET]]
constant_cols = CR.get_constant_cols(df_master)
dup_pairs = CR.get_duplicate_col_pairs(df_master)
dup_drop = set(c2 for c1, c2 in dup_pairs)
drop_cols = set(constant_cols) | dup_drop
keep_cols = [c for c in feat_cols_raw if c not in drop_cols]
print(f"\nConstant: {len(constant_cols)}, Duplicate pairs: {len(dup_pairs)} -> drop {len(dup_drop)}")
print(f"Toplam drop: {len(drop_cols)}, Kalan feature: {len(keep_cols)}")

for _df in [df_master, df_kanser, df_cftr, df_pah]:
    keep_full = [ID_COL, TARGET] + keep_cols
    _df.drop(columns=[c for c in _df.columns if c not in keep_full], inplace=True)

def find_exact_dups(panel_df, master_df, feat_cols, target):
    common_ids = set(panel_df[ID_COL]) & set(master_df[ID_COL])
    if not common_ids:
        return []
    dup_ids = []
    check_cols = feat_cols + [target]
    for vid in common_ids:
        p_row = panel_df.loc[panel_df[ID_COL] == vid, check_cols].iloc[0]
        m_rows = master_df.loc[master_df[ID_COL] == vid, check_cols]
        for _, m_row in m_rows.iterrows():
            if p_row.equals(m_row):
                dup_ids.append(vid)
                break
    return dup_ids

for name, dfp in [("KANSER", df_kanser), ("CFTR", df_cftr), ("PAH", df_pah)]:
    dup_ids = find_exact_dups(dfp, df_master, keep_cols, TARGET)
    if dup_ids:
        print(f"{name}: {len(dup_ids)} birebir-ayni satir drop")

df_kanser = df_kanser[~df_kanser[ID_COL].isin(find_exact_dups(df_kanser, df_master, keep_cols, TARGET))].reset_index(drop=True)
df_cftr = df_cftr[~df_cftr[ID_COL].isin(find_exact_dups(df_cftr, df_master, keep_cols, TARGET))].reset_index(drop=True)
df_pah = df_pah[~df_pah[ID_COL].isin(find_exact_dups(df_pah, df_master, keep_cols, TARGET))].reset_index(drop=True)

print(f"\nFinal shapes: MASTER={df_master.shape}, KANSER={df_kanser.shape}, CFTR={df_cftr.shape}, PAH={df_pah.shape}")

# ============================================================================

In [ ]:
# Cell 3: Missing-Handling Stratejileri -- M1-M5 (NB14 tanimlari, progress.md L329-333) + native_nan
# ============================================================================
# M1: Flag yok, medyan imputation
# M2: Flag yok, NaN iceren sutunlar drop
# M3: Flag (>%50 NaN) + medyan imputation (mevcut champion stratejisi, orijinal sutun korunur)
# M4: Flag (>%50 NaN) + NaN iceren sutunlar drop
# M5: Flag (>%50 NaN) -> drop, <=%50 NaN -> medyan imputation
# native_nan: Hic imputation yok, LightGBM/CatBoost'un native NaN-splitting'ine birak (flag da yok)

def fit_missing_strategy(train_df, cols, strategy, high_miss_thr=HIGH_MISS_THR):
    """Train uzerinde fit edilen strateji durumunu dondurur (kolon listeleri + medyanlar)."""
    X = train_df[cols].copy()
    cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
    num_cols = [c for c in X.columns if c not in cat_cols]
    miss_frac = X[num_cols].isnull().mean()
    high_miss = miss_frac[miss_frac > high_miss_thr].index.tolist()
    any_miss = miss_frac[miss_frac > 0].index.tolist()
    medians = X[num_cols].median()

    le_maps = {}
    for c in cat_cols:
        Xc = X[c].fillna("MISSING")
        le = LabelEncoder()
        le.fit(Xc)
        le_maps[c] = le

    if strategy == "M1":
        flag_cols, drop_cols_s, impute = [], [], True
    elif strategy == "M2":
        flag_cols, drop_cols_s, impute = [], any_miss, True
    elif strategy == "M3":
        flag_cols, drop_cols_s, impute = high_miss, [], True
    elif strategy == "M4":
        flag_cols, drop_cols_s, impute = high_miss, any_miss, True
    elif strategy == "M5":
        flag_cols, drop_cols_s, impute = high_miss, high_miss, True
        # M5: >%50 NaN -> drop (ama flag zaten bilgiyi tasir), <=%50 -> medyan impute
    elif strategy == "native_nan":
        flag_cols, drop_cols_s, impute = [], [], False
    else:
        raise ValueError(strategy)

    return {
        "cat_cols": cat_cols, "num_cols": num_cols,
        "flag_cols": flag_cols, "drop_cols": drop_cols_s,
        "impute": impute, "medians": medians, "le_maps": le_maps,
        "keep_cols": cols, "strategy": strategy,
    }

def transform_missing_strategy(df, prep):
    kc = prep["keep_cols"]
    X = df[kc].copy()
    cat_cols, num_cols = prep["cat_cols"], prep["num_cols"]

    for c in prep["flag_cols"]:
        X[f"is_missing_{c}"] = X[c].isnull().astype(int)

    if prep["impute"]:
        for c in num_cols:
            X[c] = X[c].fillna(prep["medians"][c])
    # native_nan: numeric NaN'lar oldugu gibi birakilir (LGBM/CatBoost native split eder)

    for c in cat_cols:
        X[c] = X[c].fillna("MISSING")
        le = prep["le_maps"][c]
        X[c] = X[c].apply(lambda v: le.transform([v])[0] if v in le.classes_ else -1)

    drop_now = [c for c in prep["drop_cols"] if c in X.columns]
    if drop_now:
        X = X.drop(columns=drop_now)
    return X

STRATEGIES = ["M1", "M2", "M3", "M4", "M5", "native_nan"]
print(f"Missing-handling stratejileri hazir: {STRATEGIES}")

# ============================================================================

In [ ]:
# Cell 4: Degerlendirme Altyapisi (NB39/NB44/NB46 ile birebir ayni protokol)
# ============================================================================
def _f1_pos(y, p):
    return f1_score(y, p, pos_label=1, zero_division=0)

def _resample_8020(y, prob, rng):
    y = np.asarray(y); prob = np.asarray(prob)
    neg = np.where(y == 0)[0]; pos = np.where(y == 1)[0]
    if len(neg) == 0 or len(pos) == 0:
        return y, prob
    npos = max(1, int(round(len(neg) * (1 - FINAL_BENIGN_FRAC) / FINAL_BENIGN_FRAC)))
    keep = np.concatenate([neg, rng.choice(pos, size=npos, replace=True)])
    return y[keep], prob[keep]

def bootstrap_8020(y_te, p_te, thr, n=N_BOOT):
    rng = np.random.RandomState(BOOT_SEED)
    f1s = []
    for _ in range(n):
        yb, pb = _resample_8020(y_te, p_te, rng)
        f1s.append(_f1_pos(yb, (pb >= thr).astype(int)))
    f1s = np.array(f1s)
    return {"mean": float(f1s.mean()), "std": float(f1s.std()),
            "lo": float(np.percentile(f1s, 2.5)), "hi": float(np.percentile(f1s, 97.5))}

def select_threshold_8020_robust(y, prob, n=N_BOOT):
    rng = np.random.RandomState(BOOT_SEED)
    thr_scores = {}
    for thr in np.arange(0.05, 0.95, 0.01):
        thr = round(thr, 2)
        f1s = []
        for _ in range(n):
            yb, pb = _resample_8020(y, prob, rng)
            f1s.append(_f1_pos(yb, (pb >= thr).astype(int)))
        thr_scores[thr] = np.mean(f1s)
    return float(max(thr_scores, key=thr_scores.get))

def select_threshold_raw(y, prob):
    thr_scores = {}
    for thr in np.arange(0.05, 0.95, 0.01):
        thr = round(thr, 2)
        thr_scores[thr] = _f1_pos(y, (prob >= thr).astype(int))
    return float(max(thr_scores, key=thr_scores.get))

def floor_f1(y):
    prev = float(np.mean(y))
    return 2 * prev / (1 + prev)

def eval_full(label, y_test, p_test, y_train, p_train):
    thr_raw = select_threshold_raw(y_train, p_train)
    thr_8020 = select_threshold_8020_robust(y_train, p_train)

    yp_raw = (p_test >= thr_raw).astype(int)
    f1_raw = _f1_pos(y_test, yp_raw)

    yp_8020_thr = (p_test >= thr_8020).astype(int)
    f1_5050 = _f1_pos(y_test, yp_8020_thr)
    mcc_5050 = matthews_corrcoef(y_test, yp_8020_thr)
    prec_5050 = precision_score(y_test, yp_8020_thr, pos_label=1, zero_division=0)
    rec_5050 = recall_score(y_test, yp_8020_thr, pos_label=1, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_test, yp_8020_thr, labels=[0, 1]).ravel()
    auprc = average_precision_score(y_test, p_test) if len(np.unique(y_test)) > 1 else 0.0

    boot = bootstrap_8020(y_test, p_test, thr_8020)
    f1_8020 = boot["mean"]

    rng = np.random.RandomState(BOOT_SEED)
    mccs = []
    for _ in range(N_BOOT):
        yb, pb = _resample_8020(y_test, p_test, rng)
        mccs.append(matthews_corrcoef(yb, (pb >= thr_8020).astype(int)))
    mcc_8020 = float(np.mean(mccs))

    yp_train_8020 = (p_train >= thr_8020).astype(int)
    train_f1 = _f1_pos(y_train, yp_train_8020)
    train_mcc = matthews_corrcoef(y_train, yp_train_8020)

    floor = floor_f1(y_test)

    return {
        "label": label,
        "thr_raw": thr_raw, "thr_8020": thr_8020,
        "f1_raw": f1_raw,
        "f1_5050_8020thr": f1_5050, "mcc_5050_8020thr": mcc_5050,
        "prec_5050": prec_5050, "rec_5050": rec_5050,
        "f1_8020_boot": f1_8020, "f1_8020_std": boot["std"],
        "f1_8020_ci_lo": boot["lo"], "f1_8020_ci_hi": boot["hi"],
        "mcc_8020_boot": mcc_8020,
        "auprc": auprc, "fp": int(fp), "fn": int(fn), "tp": int(tp), "tn": int(tn),
        "floor_f1": floor, "gecti_mi_floor": bool(f1_8020 > floor),
        "train_f1": train_f1, "train_mcc": train_mcc,
        "train_test_gap": train_f1 - f1_8020,
    }

print("Degerlendirme altyapisi hazir (f1_raw / f1_8020 / mcc_8020 + floor + train-gap).")

# ============================================================================

In [ ]:
# Cell 5: Model Helper'lari (LGBM native-NaN destekli, CatBoost, BalancedBagging)
# ============================================================================
LGBM_PARAMS = {
    "n_estimators": 300, "num_leaves": 31, "learning_rate": 0.05,
    "min_child_samples": 20, "subsample": 0.8, "colsample_bytree": 0.8,
    "class_weight": "balanced",
    "random_state": SEED, "verbose": -1, "n_jobs": 1, "importance_type": "gain",
}
CB_PARAMS = {
    "iterations": 300, "depth": 6, "learning_rate": 0.05,
    "auto_class_weights": "Balanced",
    "random_seed": SEED, "verbose": 0, "thread_count": 1,
}

def oof_lgbm(X_train, y_train, X_test, n_splits=5):
    """LGBM native NaN-handling destekler; imputation yapilmamis (native_nan) sutunlar dahi calisir."""
    oof = np.zeros(len(y_train))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    for tri, vai in skf.split(X_train, y_train):
        m = LGBMClassifier(**LGBM_PARAMS)
        m.fit(X_train.iloc[tri], y_train[tri])
        oof[vai] = m.predict_proba(X_train.iloc[vai])[:, 1]
    mf = LGBMClassifier(**LGBM_PARAMS)
    mf.fit(X_train, y_train)
    test_proba = mf.predict_proba(X_test)[:, 1]
    return oof, test_proba

def oof_balbag_lgbm(X_train, y_train, X_test, n_splits=5):
    """BalancedBagging base=LGBM (native NaN destekler). Dis paralellik n_jobs, ic n_jobs=1 (NB46 deadlock dersi)."""
    base = LGBMClassifier(**{**LGBM_PARAMS, "n_jobs": 1})
    oof = np.zeros(len(y_train))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    for tri, vai in skf.split(X_train, y_train):
        m = BalancedBaggingClassifier(estimator=base, n_estimators=20, random_state=SEED, n_jobs=-1)
        m.fit(X_train.iloc[tri], y_train[tri])
        oof[vai] = m.predict_proba(X_train.iloc[vai])[:, 1]
    mf = BalancedBaggingClassifier(estimator=base, n_estimators=20, random_state=SEED, n_jobs=-1)
    mf.fit(X_train, y_train)
    test_proba = mf.predict_proba(X_test)[:, 1]
    return oof, test_proba

def _prep_cat(X_df):
    cat_cols = X_df.select_dtypes(include=["object", "category"]).columns.tolist()
    Xn = X_df.copy()
    le_maps = {}
    for c in cat_cols:
        Xn[c] = Xn[c].fillna("MISSING").astype(str)
        le = LabelEncoder()
        Xn[c] = le.fit_transform(Xn[c])
        le_maps[c] = le
    return Xn, cat_cols, le_maps

def _apply_cat(X_df, cat_cols, le_maps):
    Xn = X_df.copy()
    for c in cat_cols:
        Xn[c] = Xn[c].fillna("MISSING").astype(str)
        le = le_maps[c]
        Xn[c] = Xn[c].apply(lambda v: le.transform([v])[0] if v in le.classes_ else -1)
    return Xn

def oof_catboost(X_train_df, y_train, X_test_df, n_splits=5):
    """CatBoost native NaN-handling destekler (numeric NaN'lari kendi ayirir)."""
    X_tr, cat_cols, le_maps = _prep_cat(X_train_df)
    X_te = _apply_cat(X_test_df, cat_cols, le_maps)
    cat_idx = [list(X_tr.columns).index(c) for c in cat_cols]

    oof = np.zeros(len(y_train))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    for tri, vai in skf.split(X_tr, y_train):
        m = CatBoostClassifier(**CB_PARAMS)
        m.fit(X_tr.iloc[tri], y_train[tri], cat_features=cat_idx, silent=True)
        oof[vai] = m.predict_proba(X_tr.iloc[vai])[:, 1]
    mf = CatBoostClassifier(**CB_PARAMS)
    mf.fit(X_tr, y_train, cat_features=cat_idx, silent=True)
    test_proba = mf.predict_proba(X_te)[:, 1]
    return oof, test_proba

print("Model helper'lari hazir: oof_lgbm, oof_balbag_lgbm, oof_catboost (hepsi native-NaN uyumlu).")

# ============================================================================

In [ ]:
# Cell 6: Panel-Bazli Champion Pool Insasi (NB39/NB32/NB21/NB20 reçetelerinden, SABIT)
# ============================================================================
# Her panelin mevcut champion recetesi (pool kompozisyonu + model ailesi) SABIT tutulur.
# Degisen TEK eksen: missing-handling stratejisi (Cell 3). Bu, NB44'teki "tek eksen
# degisimi" ilkesiyle ayni disiplin.

def build_reverse_pool(df_pool_source, benign_frac, seed=SEED):
    """Gercek resample ile hedef benign/pathogenic oranini uret (NB39 mantigi)."""
    y = df_pool_source[TARGET].values
    idx_ben = np.where(y == 0)[0]
    idx_pat = np.where(y == 1)[0]
    rng = np.random.RandomState(seed)
    n_ben = len(idx_ben)
    n_pat = int(round(n_ben * (1 - benign_frac) / benign_frac))
    n_pat = min(n_pat, len(idx_pat))
    pat_sample = rng.choice(idx_pat, size=n_pat, replace=False)
    idx = np.concatenate([idx_ben, pat_sample])
    rng.shuffle(idx)
    return df_pool_source.iloc[idx].reset_index(drop=True)

def panel_5050_split(df, seed=SEED):
    pos = df[df[TARGET] == 1].sample(frac=1.0, random_state=seed)
    neg = df[df[TARGET] == 0].sample(frac=1.0, random_state=seed)
    npos = int(round(len(pos) * PANEL_SPLIT_FRAC))
    nneg = int(round(len(neg) * PANEL_SPLIT_FRAC))
    tr = pd.concat([pos.iloc[:npos], neg.iloc[:nneg]]).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    te = pd.concat([pos.iloc[npos:], neg.iloc[nneg:]]).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return tr, te

# --- MASTER: champion = S1_6040/balbag (NB39), model=BalBag(LGBM), pool=MASTER kendi train'i reverse-resample ---
master_train, master_test = panel_5050_split(df_master)
master_pool = build_reverse_pool(master_train, benign_frac=0.60)
print(f"MASTER pool (S1_6040): n={len(master_pool)}, benign_frac={(master_pool[TARGET]==0).mean():.3f}")
print(f"MASTER test: n={len(master_test)} (pos={master_test[TARGET].sum()}, neg={(master_test[TARGET]==0).sum()})")

# --- KANSER: champion = P9_REVERSE_6040/catboost/with_fe (NB32), pool=COMBINED(MASTER+PAH+CFTR) reverse-resample ---
kanser_train, kanser_test = panel_5050_split(df_kanser)
df_combined_for_kanser = pd.concat([df_master, df_pah, df_cftr], ignore_index=True)
kanser_pool = build_reverse_pool(df_combined_for_kanser, benign_frac=0.60)
print(f"KANSER pool (P9_REVERSE_6040): n={len(kanser_pool)}, benign_frac={(kanser_pool[TARGET]==0).mean():.3f}")
print(f"KANSER test: n={len(kanser_test)} (pos={kanser_test[TARGET].sum()}, neg={(kanser_test[TARGET]==0).sum()})")

# --- PAH: champion = P4_COMBINED_BalBag (NB21), pool=COMBINED(MASTER+KANSER+CFTR), orijinal dagilim ---
pah_train, pah_test = panel_5050_split(df_pah)
pah_pool = pd.concat([df_master, df_kanser, df_cftr], ignore_index=True)
print(f"PAH pool (COMBINED): n={len(pah_pool)}, benign_frac={(pah_pool[TARGET]==0).mean():.3f}")
print(f"PAH test: n={len(pah_test)} (pos={pah_test[TARGET].sum()}, neg={(pah_test[TARGET]==0).sum()})")

# --- CFTR: champion = S0c_COMBINED (NB20), pool=COMBINED(MASTER+KANSER+PAH), orijinal dagilim ---
cftr_train, cftr_test = panel_5050_split(df_cftr)
cftr_pool = pd.concat([df_master, df_kanser, df_pah], ignore_index=True)
print(f"CFTR pool (COMBINED): n={len(cftr_pool)}, benign_frac={(cftr_pool[TARGET]==0).mean():.3f}")
print(f"CFTR test: n={len(cftr_test)} (pos={cftr_test[TARGET].sum()}, neg={(cftr_test[TARGET]==0).sum()})")

# ============================================================================

In [ ]:
# Cell 7: FE Yardimcisi (KANSER champion "with_fe" iceriyor -- NB16/NB32/NB44 ile ayni)
# ============================================================================
STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")
_GRANTHAM = {
 ('S','R'):110,('S','L'):145,('S','P'):74,('S','T'):58,('S','A'):99,('S','V'):124,
 ('S','G'):56,('S','I'):142,('S','F'):155,('S','Y'):144,('S','C'):112,('S','H'):89,
 ('S','Q'):68,('S','N'):46,('S','K'):121,('S','D'):65,('S','E'):80,('S','M'):135,('S','W'):177,
 ('R','L'):102,('R','P'):103,('R','T'):71,('R','A'):112,('R','V'):96,('R','G'):125,('R','I'):97,
 ('R','F'):97,('R','Y'):77,('R','C'):180,('R','H'):29,('R','Q'):43,('R','N'):86,('R','K'):26,
 ('R','D'):96,('R','E'):54,('R','M'):91,('R','W'):101,
 ('L','P'):98,('L','T'):92,('L','A'):96,('L','V'):32,('L','G'):138,('L','I'):5,('L','F'):22,
 ('L','Y'):36,('L','C'):198,('L','H'):99,('L','Q'):113,('L','N'):153,('L','K'):107,('L','D'):172,
 ('L','E'):138,('L','M'):15,('L','W'):61,
 ('P','T'):38,('P','A'):27,('P','V'):68,('P','G'):42,('P','I'):95,('P','F'):114,('P','Y'):110,
 ('P','C'):169,('P','H'):77,('P','Q'):76,('P','N'):91,('P','K'):103,('P','D'):108,('P','E'):93,
 ('P','M'):87,('P','W'):147,
 ('T','A'):58,('T','V'):69,('T','G'):59,('T','I'):89,('T','F'):103,('T','Y'):92,('T','C'):149,
 ('T','H'):47,('T','Q'):42,('T','N'):65,('T','K'):78,('T','D'):85,('T','E'):65,('T','M'):81,('T','W'):128,
 ('A','V'):64,('A','G'):60,('A','I'):94,('A','F'):113,('A','Y'):112,('A','C'):195,('A','H'):86,
 ('A','Q'):91,('A','N'):111,('A','K'):106,('A','D'):126,('A','E'):107,('A','M'):84,('A','W'):148,
 ('V','G'):109,('V','I'):29,('V','F'):50,('V','Y'):55,('V','C'):192,('V','H'):84,('V','Q'):96,
 ('V','N'):133,('V','K'):97,('V','D'):152,('V','E'):121,('V','M'):21,('V','W'):88,
 ('G','I'):135,('G','F'):153,('G','Y'):147,('G','C'):159,('G','H'):98,('G','Q'):87,('G','N'):80,
 ('G','K'):127,('G','D'):94,('G','E'):98,('G','M'):127,('G','W'):184,
 ('I','F'):21,('I','Y'):33,('I','C'):198,('I','H'):94,('I','Q'):109,('I','N'):149,('I','K'):102,
 ('I','D'):168,('I','E'):134,('I','M'):10,('I','W'):61,
 ('F','Y'):22,('F','C'):205,('F','H'):100,('F','Q'):116,('F','N'):158,('F','K'):102,('F','D'):177,
 ('F','E'):140,('F','M'):28,('F','W'):40,
 ('Y','C'):194,('Y','H'):83,('Y','Q'):99,('Y','N'):143,('Y','K'):85,('Y','D'):160,('Y','E'):122,
 ('Y','M'):36,('Y','W'):37,
 ('C','H'):174,('C','Q'):154,('C','N'):139,('C','K'):202,('C','D'):154,('C','E'):170,('C','M'):196,('C','W'):215,
 ('H','Q'):24,('H','N'):68,('H','K'):32,('H','D'):81,('H','E'):40,('H','M'):87,('H','W'):115,
 ('Q','N'):46,('Q','K'):53,('Q','D'):61,('Q','E'):29,('Q','M'):101,('Q','W'):130,
 ('N','K'):94,('N','D'):23,('N','E'):42,('N','M'):142,('N','W'):174,
 ('K','D'):101,('K','E'):56,('K','M'):95,('K','W'):110,
 ('D','E'):45,('D','M'):160,('D','W'):181,
 ('E','M'):126,('E','W'):152,
 ('M','W'):67,
}
_B62_RAW = '''A4 R-1 N-2 D-2 C0 Q-1 E-1 G0 H-2 I-1 L-1 K-1 M-1 F-2 P-1 S1 T0 W-3 Y-2 V0
R5 N0 D-2 C-3 Q1 E0 G-2 H0 I-3 L-2 K2 M-1 F-3 P-2 S-1 T-1 W-3 Y-2 V-3
N6 D1 C-3 Q0 E0 G0 H1 I-3 L-3 K0 M-2 F-3 P-2 S1 T0 W-4 Y-2 V-3
D6 C-3 Q0 E2 G-1 H-1 I-3 L-4 K-1 M-3 F-3 P-1 S0 T-1 W-4 Y-3 V-3
C9 Q-3 E-4 G-3 H-3 I-1 L-1 K-3 M-1 F-2 P-3 S-1 T-1 W-2 Y-2 V-1
Q5 E2 G-2 H0 I-3 L-2 K1 M0 F-3 P-1 S0 T-1 W-2 Y-1 V-2
E5 G-2 H0 I-3 L-3 K1 M-2 F-3 P-1 S0 T-1 W-3 Y-2 V-2
G6 H-2 I-4 L-4 K-2 M-3 F-3 P-2 S0 T-2 W-2 Y-3 V-3
H8 I-3 L-3 K-1 M-2 F-1 P-2 S-1 T-2 W-2 Y2 V-3
I4 L2 K-3 M1 F0 P-3 S-2 T-1 W-3 Y-1 V3
L4 K-2 M2 F0 P-3 S-2 T-1 W-2 Y-1 V1
K5 M-1 F-3 P-1 S0 T-1 W-3 Y-2 V-2
M5 F0 P-2 S-1 T-1 W-1 Y-1 V1
F6 P-4 S-2 T-2 W1 Y3 V-1
P7 S-1 T-1 W-4 Y-3 V-2
S4 T1 W-3 Y-2 V-2
T5 W-2 Y-2 V0
W11 Y2 V-3
Y7 V-1
V4'''
_ORDER = list("ARNDCQEGHILKMFPSTWYV")
_B62 = {}
for ri, line in enumerate(_B62_RAW.strip().split("\n")):
    toks = line.split()
    row_aa = toks[0][0]
    vals = [toks[0][1:]] + toks[1:]
    for ci, tok in enumerate(vals):
        col_aa = _ORDER[ri + ci]
        v = int(tok[1:] if tok[0].isalpha() else tok)
        _B62[(row_aa, col_aa)] = v; _B62[(col_aa, row_aa)] = v

def grantham(a, b):
    if a == b: return 0
    return _GRANTHAM.get((a, b)) or _GRANTHAM.get((b, a))

def blosum62(a, b):
    return _B62.get((a, b), 0)

def add_fe(df):
    out = df.copy()
    a1 = out["AA_1"].astype("object"); a2 = out["AA_2"].astype("object")
    out["fe_aa_stopgain"] = (a2 == "*").astype(int)
    def _nonstd(v):
        return 0 if (isinstance(v, str) and v in STANDARD_AA) else 1
    out["fe_aa_nonstandard"] = (a1.map(_nonstd) | a2.map(_nonstd)).astype(int)
    def _gr(r):
        x, y = r["AA_1"], r["AA_2"]
        if isinstance(x, str) and isinstance(y, str) and x in STANDARD_AA and y in STANDARD_AA:
            return grantham(x, y)
        return -1
    def _bl(r):
        x, y = r["AA_1"], r["AA_2"]
        if isinstance(x, str) and isinstance(y, str) and x in STANDARD_AA and y in STANDARD_AA:
            return blosum62(x, y)
        return 0
    out["fe_grantham"] = out.apply(_gr, axis=1).astype(float)
    out["fe_blosum62"] = out.apply(_bl, axis=1).astype(float)
    return out

FE_NEW_COLS = ["fe_aa_stopgain", "fe_aa_nonstandard", "fe_grantham", "fe_blosum62"]
print(f"FE hazir (KANSER champion icin). Yeni sutunlar: {FE_NEW_COLS}")

# ============================================================================

In [ ]:
# Cell 8: Panel Deney Calistirici -- her panelde champion model + 6 missing-strategi
# ============================================================================
PANEL_CONFIGS = OrderedDict([
    ("MASTER", {"pool": master_pool, "test": master_test, "model_fn": "balbag", "use_fe": False}),
    ("KANSER", {"pool": kanser_pool, "test": kanser_test, "model_fn": "catboost", "use_fe": True}),
    ("PAH",    {"pool": pah_pool,    "test": pah_test,    "model_fn": "balbag", "use_fe": False}),
    ("CFTR",   {"pool": cftr_pool,   "test": cftr_test,   "model_fn": "lgbm", "use_fe": False}),
])

MODEL_FNS = {"lgbm": oof_lgbm, "balbag": oof_balbag_lgbm, "catboost": oof_catboost}

all_results = []
for panel_name, cfg in PANEL_CONFIGS.items():
    print(f"\n{'='*70}\nPANEL: {panel_name} (model={cfg['model_fn']}, use_fe={cfg['use_fe']})\n{'='*70}")
    pool_df, test_df = cfg["pool"], cfg["test"]
    model_fn = MODEL_FNS[cfg["model_fn"]]

    if cfg["use_fe"]:
        pool_df = add_fe(pool_df)
        test_df = add_fe(test_df)
        strat_cols = keep_cols + FE_NEW_COLS
    else:
        strat_cols = keep_cols

    for strategy in STRATEGIES:
        prep = fit_missing_strategy(pool_df, strat_cols, strategy)
        X_pool = transform_missing_strategy(pool_df, prep)
        X_test = transform_missing_strategy(test_df, prep)
        y_pool = pool_df[TARGET].values
        y_test = test_df[TARGET].values

        n_flags = len(prep["flag_cols"])
        n_dropped = len([c for c in prep["drop_cols"] if c in strat_cols])
        n_features = X_pool.shape[1]

        if cfg["model_fn"] == "catboost" and not HAS_CATBOOST:
            print(f"  [{strategy}] catboost yok, atlaniyor")
            continue

        oof, test_proba = model_fn(X_pool, y_pool, X_test)
        res = eval_full(f"{panel_name}/{strategy}", y_test, test_proba, y_pool, oof)
        res.update({
            "panel": panel_name, "strategy": strategy, "model": cfg["model_fn"],
            "use_fe": cfg["use_fe"], "n_flags": n_flags, "n_dropped": n_dropped,
            "n_features": n_features, "n_pool": len(y_pool), "n_test": len(y_test),
        })
        all_results.append(res)
        print(f"  [{strategy}] n_feat={n_features} flags={n_flags} dropped={n_dropped} "
              f"f1_8020={res['f1_8020_boot']:.4f} mcc_8020={res['mcc_8020_boot']:.4f} "
              f"floor={res['floor_f1']:.4f} gap={res['train_test_gap']:.4f}")

results_df = pd.DataFrame(all_results)
results_df.to_csv(os.path.join(RESULTS_DIR, "nb48_missing_handling_rejudge_results.csv"), index=False)
print(f"\nSonuclar kaydedildi: {os.path.join(RESULTS_DIR, 'nb48_missing_handling_rejudge_results.csv')}")

# ============================================================================

In [ ]:
# Cell 9: Panel-Bazli Kazanan Strateji + Champion Karsilastirmasi
# ============================================================================
CHAMPIONS = {
    "MASTER": {"name": "S1_6040_balbag_NB39", "boot_f1": 0.638, "strategy_used": "M3"},
    "KANSER": {"name": "P9_REVERSE_6040_catboost_with_fe_NB32", "boot_f1": 0.730, "strategy_used": "M3"},
    "PAH":    {"name": "P4_COMBINED_BalBag_NB21", "boot_f1": 0.582, "strategy_used": "M3"},
    "CFTR":   {"name": "S0c_PriorShift_NB20", "boot_f1": 0.863, "strategy_used": "M3"},
}

summary_rows = []
for panel_name in PANEL_CONFIGS.keys():
    sub = results_df[results_df["panel"] == panel_name].copy()
    if sub.empty:
        continue
    best_row = sub.loc[sub["f1_8020_boot"].idxmax()]
    champ = CHAMPIONS[panel_name]
    delta_vs_champion_m3 = best_row["f1_8020_boot"] - champ["boot_f1"]
    m3_row = sub[sub["strategy"] == "M3"]
    m3_f1 = float(m3_row["f1_8020_boot"].iloc[0]) if not m3_row.empty else None
    summary_rows.append({
        "panel": panel_name,
        "best_strategy": best_row["strategy"],
        "best_f1_8020": best_row["f1_8020_boot"],
        "best_ci": f"[{best_row['f1_8020_ci_lo']:.3f}-{best_row['f1_8020_ci_hi']:.3f}]",
        "m3_f1_8020_repro": m3_f1,
        "reference_champion": champ["name"],
        "reference_boot_f1": champ["boot_f1"],
        "delta_best_vs_reference": delta_vs_champion_m3,
        "floor_f1": best_row["floor_f1"],
        "gecti_mi_floor": best_row["gecti_mi_floor"],
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(RESULTS_DIR, "nb48_panel_summary.csv"), index=False)
print("\n" + "="*70)
print("PANEL-BAZLI KAZANAN MISSING-STRATEJI OZETI")
print("="*70)
print(summary_df.to_string(index=False))

# ============================================================================

In [ ]:
# Cell 10: Gorsellestirme -- panel x strateji F1(8020) karsilastirma
# ============================================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, panel_name in zip(axes.flat, PANEL_CONFIGS.keys()):
    sub = results_df[results_df["panel"] == panel_name].sort_values("f1_8020_boot", ascending=False)
    colors = ["#2ca02c" if s == sub.iloc[0]["strategy"] else "#1f77b4" for s in sub["strategy"]]
    ax.bar(sub["strategy"], sub["f1_8020_boot"], yerr=sub["f1_8020_std"], color=colors, capsize=4)
    ax.axhline(sub["floor_f1"].iloc[0], color="red", linestyle="--", label="floor-F1")
    ax.axhline(CHAMPIONS[panel_name]["boot_f1"], color="orange", linestyle=":", label="mevcut champion (M3)")
    ax.set_title(f"{panel_name} — Missing-Handling Karsilastirmasi")
    ax.set_ylabel("Boot-F1 (%80/20)")
    ax.legend(fontsize=8)
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, "nb48_missing_handling_comparison.png")
plt.savefig(fig_path, dpi=120)
plt.close()
print(f"\nGrafik kaydedildi: {fig_path}")

# ============================================================================

In [ ]:
# Cell 11: Ozet JSON
# ============================================================================
summary_json = {
    "experiment": "NB48_missing_handling_rejudge",
    "hypothesis": "MASTER'da flag'li (M3/M4/M5) kazanir (MNAR), PAH'ta flag'siz (M1/M2/native_nan) kazanir (MCAR)",
    "strategies_tested": STRATEGIES,
    "panel_configs": {k: {"model": v["model_fn"], "use_fe": v["use_fe"], "n_pool": len(v["pool"])} for k, v in PANEL_CONFIGS.items()},
    "results_summary": summary_df.to_dict(orient="records"),
    "champions_reference": CHAMPIONS,
}
with open(os.path.join(RESULTS_DIR, "nb48_summary.json"), "w") as f:
    json.dump(summary_json, f, indent=2, default=str)
print(f"\nOzet JSON kaydedildi: {os.path.join(RESULTS_DIR, 'nb48_summary.json')}")
print("\nNB48 TAMAMLANDI.")